# Metric Presentation and Visualization
## Necessary packages and functions call

- DDPM-TS: Interpretable Diffusion for Time Series Generation
- Metrics: 
    - discriminative_metrics
    - predictive_metrics
    - visualization

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

from Utils.metric_utils import display_scores
from Utils.discriminative_metric import discriminative_score_metrics
from Utils.predictive_metric import predictive_score_metrics

## Data Loading

Load original dataset and preprocess the loaded data.

In [2]:
iterations = 5
dataset_name = 'energy'
seq_length = 169
# ori_data = np.load('../toy_exp/samples/energy_ground_truth_24_train.npy')
# ori_data = np.load(f'../energy_results/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
ori_data = np.load(f'../energy_results/ETTh1/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')
fake_data = np.load('../energy_results/ETTh1/ddpm_fake_energy_0_to_1.npy')

## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [3]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('energy:')
display_scores(discriminative_score)
print()

Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Please use tf.global_variables instead.


training: 100%|██████████| 2000/2000 [00:48<00:00, 40.99it/s]


Iter 0:  0.14258879781420764 , 0.7400956284153005 , 0.5450819672131147 



training: 100%|██████████| 2000/2000 [00:49<00:00, 40.10it/s]


Iter 1:  0.15761612021857918 , 0.6263661202185792 , 0.6888661202185792 



...

Iter 2:  0.14088114754098358 , 0.7216530054644809 , 0.5601092896174863 



training: 100%|██████████| 2000/2000 [00:49<00:00, 40.46it/s]


Iter 3:  0.14480874316939896 , 0.6786202185792349 , 0.6109972677595629 



training: 100%|██████████| 2000/2000 [00:49<00:00, 40.80it/s]


Iter 4:  0.1335382513661202 , 0.7609289617486339 , 0.5061475409836066 

energy:
Final Score:  0.1438866120218579 ± 0.010880608662896678



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [4]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('energy:')
display_scores(predictive_score)
print()

training: 100%|██████████| 5000/5000 [01:27<00:00, 57.35it/s]


0  epoch:  0.04994972024476305 



training: 100%|██████████| 5000/5000 [01:27<00:00, 57.45it/s]


1  epoch:  0.052155866902452816 



training: 100%|██████████| 5000/5000 [01:26<00:00, 57.70it/s]


2  epoch:  0.05414841038579742 



training: 100%|██████████| 5000/5000 [01:23<00:00, 60.22it/s]


3  epoch:  0.05550577603357332 



training: 100%|██████████| 5000/5000 [01:23<00:00, 59.92it/s]


4  epoch:  0.05356895250425979 

energy:
Final Score:  0.05306574521416928 ± 0.0026278596044391972



In [5]:
import os
import pandas as pd
import numpy as np
import scipy.stats

def compute_score(results, confidence=0.95):
    """Return mean ± CI string like display_scores."""
    mean = np.mean(results)
    sigma = scipy.stats.sem(results)
    sigma = sigma * scipy.stats.t.ppf((1 + confidence) / 2., len(results)-1)
    return f"{mean:.4f} ± {sigma:.4f}"

# Example usage after your iterations
disc_result = compute_score(discriminative_score)
pred_result = compute_score(predictive_score)

results = {
    "Metric": ["Discriminative", "Predictive"],
    "Result": [disc_result, pred_result]
}

results_df = pd.DataFrame(results)

save_folder = "../figures/"
os.makedirs(save_folder, exist_ok=True)

save_path = os.path.join(save_folder, f"ETTh1_{dataset_name}_metrics.csv")
results_df.to_csv(save_path, index=False)

print(f"✅ Compact metrics saved to {save_path}")

✅ Compact metrics saved to ../figures/ETTh1_energy_metrics.csv
